# Layer 2 Architecture Exploration (FPN vs DeepLabV3Plus vs Unet++)

This notebook compares segmentation architectures while keeping the backbone fixed.

Fixed design:
- Backbone: `resnet34`
- Encoder weights: `imagenet`
- Architectures: `FPN`, `DeepLabV3Plus`, `UnetPlusPlus` (this is the argument name used by `train_ttd.py`)
- Settings: `Single-TB`, `Shift-TA_TC-to-TB_10pct`, `Shift-TA_TB-to-TC_100pct`


## 1. Setup

In [8]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
SCRIPT_DIR = PROJECT_ROOT / 'baseline_models_scripts'
RESULTS_DIR = SCRIPT_DIR / 'runs'
VENV_PY = Path('/users/7/yu001011/csci5527/.venv/bin/python')

PROJECT_ROOT, SCRIPT_DIR, RESULTS_DIR, VENV_PY

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 PosixPath('/users/7/yu001011/csci5527/.venv/bin/python'))

## 2. Experiment Design

In [9]:
BACKBONE = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 512
LR = 1e-4

SETTINGS = [
#     'Single-TB',
#     'Shift-TA_TC-to-TB_10pct',
    'Shift-TA_TB-to-TC_10pct',
]

ARCHES = [
    'FPN',
    'DeepLabV3Plus',
    'UnetPlusPlus',
]

pd.DataFrame([
    {'Setting': setting, 'Architecture': arch}
    for setting in SETTINGS
    for arch in ARCHES
])

,Setting,Architecture
0,Shift-TA_TB-to-TC_10pct,FPN
1,Shift-TA_TB-to-TC_10pct,DeepLabV3Plus
2,Shift-TA_TB-to-TC_10pct,UnetPlusPlus


## 3. Helper Functions

In [10]:
def result_filename(arch: str) -> str:
    safe_arch = arch.replace('+', 'plus').replace(' ', '_').lower()
    return f'layer2_arch_{safe_arch}_{BACKBONE}.csv'

def build_command(setting: str, arch: str):
    return [
        str(VENV_PY),
        'train_ttd.py',
        '--project-root', str(PROJECT_ROOT),
        '--experiment', setting,
        '--arch', arch,
        '--encoder', BACKBONE,
        '--encoder-weights', ENCODER_WEIGHTS,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--img-size', str(IMG_SIZE),
        '--lr', str(LR),
        '--results-name', result_filename(arch),
    ]

def run_one(setting: str, arch: str):
    cmd = build_command(setting, arch)
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, cwd=SCRIPT_DIR, check=True)

def run_all(settings, arches):
    for arch in arches:
        for setting in settings:
            run_one(setting, arch)


## 4. Preview Commands

In [11]:
command_preview = pd.DataFrame([
    {
        'Setting': setting,
        'Architecture': arch,
        'Command': ' '.join(build_command(setting, arch)),
    }
    for setting in SETTINGS
    for arch in ARCHES
])

display(command_preview)

,Setting,Architecture,Command
0,Shift-TA_TB-to-TC_10pct,FPN,/users/7/yu001011/csci5527/.venv/bin/python tr...
1,Shift-TA_TB-to-TC_10pct,DeepLabV3Plus,/users/7/yu001011/csci5527/.venv/bin/python tr...
2,Shift-TA_TB-to-TC_10pct,UnetPlusPlus,/users/7/yu001011/csci5527/.venv/bin/python tr...


## 5. Run a Single Experiment

In [12]:
TEST_SETTING = 'Single-TB'
TEST_ARCH = 'Unet'

# Uncomment to run:
# run_one(TEST_SETTING, TEST_ARCH)

## 6. Run the Full Layer 2 Grid

In [ ]:
# Uncomment to run all experiments:
run_all(SETTINGS, ARCHES)

Running: /users/7/yu001011/csci5527/.venv/bin/python train_ttd.py --project-root /users/7/yu001011/csci5527/CSCI5527-final --experiment Shift-TA_TB-to-TC_10pct --arch FPN --encoder resnet34 --encoder-weights imagenet --epochs 100 --batch-size 16 --img-size 512 --lr 0.0001 --results-name layer2_arch_fpn_resnet34.csv
Python executable: /users/7/yu001011/csci5527/.venv/bin/python
Torch version: 2.11.0+cu126
CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA H100
Using device: cuda:0

===== Running Shift-TA_TB-to-TC_10pct | arch=FPN | encoder=resnet34 =====
Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 38/38 [00:00<00:00, 3851.24it/s]


Sanitizing test masks...
Generating Dataloaders...

Pipeline Ready:
 - Training batches: 50
 - Validation batches: 15
 - Testing batches: 3
  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:   0%|          | 0/100 [00:11<?, ?it/s, IoU=0.2702, F1=0.4072]Epoch 0: T-Loss: 8.5231 | V-Loss: 6.9508 | IoU: 0.2702 | F1: 0.4072 [Saved Best Model]
  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:   1%|          | 1/100 [00:23<19:12, 11.65s/it, IoU=0.3681, F1=0.5243]Epoch 1: T-Loss: 6.8563 | V-Loss: 6.3048 | IoU: 0.3681 | F1: 0.5243 [Saved Best Model]
  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:   2%|▏         | 2/100 [00:34<18:52, 11.56s/it, IoU=0.4235, F1=0.5820]Epoch 2: T-Loss: 6.4673 | V-Loss: 6.1062 | IoU: 0.4235 | F1: 0.5820 [Saved Best Model]
  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:   3%|▎         | 3/100 [00:45<18:29, 11.44s/it, IoU=0.4586, F1=0.6178]Epoch 3: T-Loss: 6.3150 | V-Loss: 6.0337 | IoU: 0.4586 | F1: 0.6178 [Saved Best Model]
  → FPN-resnet34-imagenet_Shift-TA_T

  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:  82%|████████▏ | 82/100 [15:42<03:24, 11.38s/it, IoU=0.5571, F1=0.7055]Epoch 82: T-Loss: 5.4778 | V-Loss: 5.4588 | IoU: 0.5571 | F1: 0.7055


  → FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct:  87%|████████▋ | 87/100 [16:28<02:30, 11.55s/it, IoU=0.5518, F1=0.7008]

## 7. Load Architecture Results

In [ ]:
def load_arch_results(arches):
    frames = []
    for arch in arches:
        path = RESULTS_DIR / result_filename(arch)
        if path.exists():
            df = pd.read_csv(path)
            df['Architecture'] = arch
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

arch_df = load_arch_results(ARCHES)
arch_df

## 8. Pivot Table for Comparison

In [ ]:
if not arch_df.empty:
    arch_df = arch_df.copy()
    arch_df['Setting'] = arch_df['Experiment'].str.replace(r'^.+?_.+?_', '', regex=True)
    pivot_iou = arch_df.pivot(index='Setting', columns='Architecture', values='Test_IoU')
    pivot_f1 = arch_df.pivot(index='Setting', columns='Architecture', values='Test_F1')
    display(Markdown('### Test IoU'))
    display(pivot_iou)
    display(Markdown('### Test F1'))
    display(pivot_f1)
else:
    display(Markdown('No results available yet.'))